In [0]:
dbutils.widgets.text("catalog", "dev")
catalog = dbutils.widgets.get("catalog")

In [0]:
from pyspark.sql import functions as F, Window

In [0]:
region_clean = (
    spark.table(f"{catalog}.bronze.region")
    .select(
        F.col("r_regionkey").alias("r_regionkey"),
        F.trim(F.upper(F.col("r_name"))).alias("r_name"),
        F.trim(F.col("r_comment")).alias("r_comment"),
        F.col("_source_file"),
        F.col("_ingested_at"),
        F.col("_job_run_id"),
    )
)

# Rank records within each PK by _ingested_at (latest = rank 1)
from pyspark.sql.window import Window

w = Window.partitionBy("r_regionkey").orderBy(F.col("_ingested_at").desc())

region_ranked = region_clean.withColumn("_pk_rank", F.row_number().over(w))

# Flag with quarantine reason
region_flagged = (
    region_ranked
    .withColumn(
        "quarantine_reason",
        F.when(F.col("r_regionkey").isNull(), "NULL primary key")
        .when(F.col("_pk_rank") > 1, "Duplicate primary key - older record")
        .otherwise(None)
    )
    .drop("_pk_rank")
)

# Split
clean = region_flagged.filter(F.col("quarantine_reason").isNull()).drop("quarantine_reason", "_ingested_at", "_source_file", "_job_run_id")
quarantined = region_flagged.filter(F.col("quarantine_reason").isNotNull())

# Write clean records to silver.region
(clean.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.silver.region"))

# Write quarantined records (with reason) to silver.region_quarantine
(quarantined.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.silver.region_quarantine"))

# --- Report ---
total = region_clean.count()
clean_count = clean.count()
quarantine_count = quarantined.count()

print(f"Total records:      {total}")
print(f"✅ Clean → silver.region:               {clean_count}")
print(f"❌ Quarantined → silver.region_quarantine: {quarantine_count}")

# Show clean result
spark.table(f"{catalog}.silver.region").display()

In [0]:
from pyspark.sql.window import Window

# Clean and standardize nation table from bronze → silver
nation_clean = (
    spark.table(f"{catalog}.bronze.nation")
    .select(
        F.col("n_nationkey").alias("n_nationkey"),
        F.trim(F.upper(F.col("n_name"))).alias("n_name"),
        F.col("n_regionkey").alias("n_regionkey"),
        F.trim(F.col("n_comment")).alias("n_comment"),
        F.col("_source_file"),
        F.col("_ingested_at"),
        F.col("_job_run_id"),
    )
)

# --- PK validation: n_nationkey must be NOT NULL and UNIQUE ---
# Rank records within each PK by _ingested_at (latest = rank 1)
w = Window.partitionBy("n_nationkey").orderBy(F.col("_ingested_at").desc())
nation_ranked = nation_clean.withColumn("_pk_rank", F.row_number().over(w))

# --- FK validation: n_regionkey must exist in silver.region.r_regionkey ---
valid_region_keys = spark.table(f"{catalog}.silver.region").select(
    F.col("r_regionkey").alias("_valid_regionkey")
)

# Flag with quarantine reason (None = clean)
nation_flagged = (
    nation_ranked
    .join(valid_region_keys, F.col("n_regionkey") == F.col("_valid_regionkey"), "left")
    .withColumn(
        "quarantine_reason",
        F.when(F.col("n_nationkey").isNull(), "NULL primary key")
        .when(F.col("_pk_rank") > 1, "Duplicate primary key - older record")
        .when(F.col("n_regionkey").isNull(), "NULL foreign key (n_regionkey)")
        .when(F.col("_valid_regionkey").isNull(), "Foreign key not found in parent (silver.region)")
        .otherwise(None)
    )
    .drop("_pk_rank", "_valid_regionkey")
)

# Split: clean → silver, violations → quarantine
clean = (
    nation_flagged.filter(F.col("quarantine_reason").isNull())
    .drop("quarantine_reason", "_ingested_at", "_source_file", "_job_run_id")
)
quarantined = nation_flagged.filter(F.col("quarantine_reason").isNotNull())

# Write clean records to silver.nation
(clean.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.silver.nation"))

# Write quarantined records (with reason) to silver.nation_quarantine
(quarantined.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.silver.nation_quarantine"))

# --- Report ---
total = nation_clean.count()
clean_count = clean.count()
quarantine_count = quarantined.count()

print(f"Total records:      {total}")
print(f"✅ Clean → silver.nation:                  {clean_count}")
print(f"❌ Quarantined → silver.nation_quarantine:    {quarantine_count}")

# Show clean result
spark.table(f"{catalog}.silver.nation").display()

In [0]:
passenger_silver = (
    spark.read.table(f"{catalog}.bronze.passenger")
    .select(
        F.col("pa_passengerid").alias("pa_passengerid"),
        F.initcap(F.trim(F.col("pa_name"))).alias("pa_name"),
        F.initcap(F.trim(F.col("pa_address"))).alias("pa_address"),
        F.col("pa_nationkey").alias("pa_nationkey"),
        F.col("pa_phone").alias("pa_phone"),
        F.col("pa_loyaltypoints"),
        F.upper(F.trim(F.col("pa_travelpurpose"))).alias("pa_travelpurpose"),
        F.col("pa_comment"),
        F.col("_ingested_at")
    ).dropDuplicates()
)

# Ranking the passenger records to find out the latest records of the passenger
w = Window.partitionBy("pa_passengerid").orderBy(F.col("_ingested_at").desc())
passenger_ranked = passenger_silver.withColumn("_pk_rank", F.row_number().over(w))

valid_nation_keys = spark.table(f"{catalog}.silver.nation").select(F.col("n_nationkey").alias("_valid_nationkey"))

passenger_flagged = (
    passenger_ranked
    .join(valid_nation_keys, F.col("pa_nationkey") == F.col("_valid_nationkey"), "left")
    .withColumn(
        "quarantine_reason",
        F.when(F.col("pa_passengerid").isNull(), "NULL primary key")
        .when(F.col("_pk_rank") > 1, "Duplicate primary key - older record")
        .when(F.col("pa_nationkey").isNull(), "NULL FK (pa_nationkey)")
        .when(F.col("_valid_nationkey").isNull(), "Foreign key not found in parent")
        .when(F.col("pa_name").isNull(), "NULL pa_name")
        .otherwise(None)
    ).drop("_pk_rank", "_valid_nationkey")
)

clean = (
    passenger_flagged.filter(F.col("quarantine_reason").isNull())
    .drop("quarantine_reason", "_ingested_at", "_source_file", "_job_run_id")
)

quarantined = passenger_flagged.filter(F.col("quarantine_reason").isNotNull())

(
    clean.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.silver.passenger")
)

(
    quarantined.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.silver.passenger_quarantine")
)

In [0]:
airline_silver = (
    spark.read.table(f"{catalog}.bronze.airline")
    .select(
        F.col("al_airlineid").alias("al_airlineid"),
        F.initcap(F.trim(F.col("al_name"))).alias("al_name"),
        F.initcap(F.trim(F.col("al_address"))).alias("al_address"),
        F.col("al_nationkey").alias("al_nationkey"),
        F.col("al_phone").alias("al_phone"),
        F.col("al_rating").alias("al_rating"),
        F.col("al_comment").alias("al_comment"),
        F.col("_source_file"),
        F.col("_ingested_at"),
        F.col("_job_run_id")
    ).dropDuplicates()
)

w = Window.partitionBy("al_airlineid").orderBy(F.col("_ingested_at").desc())
airline_ranked = airline_silver.withColumn("_pk_rank", F.row_number().over(w))

valid_nation_keys = spark.table(f"{catalog}.silver.nation").select(F.col("n_nationkey").alias("_valid_nationkey"))

airline_flagged = (
    airline_ranked
    .join(valid_nation_keys, F.col("al_nationkey") == F.col("_valid_nationkey"), "left")
    .withColumn(
        "quarantine_reason",
        F.when(F.col("al_airlineid").isNull(), "NULL Primary key")
        .when(F.col("_pk_rank") > 1, "Duplicate primary key - older record")
        .when(F.col("al_nationkey").isNull(), "NULL FK (pa_nationkey)")
        .when(F.col("_valid_nationkey").isNull(), "Foreign key not found in parent")
    ).drop("_pk_rank", "_valid_nationkey")
)

clean = airline_flagged.filter(F.col("quarantine_reason").isNull()).drop("quarantine_reason", "_ingested_at", "source_file", "_job_run_id")

quarantined = airline_flagged.filter(F.col("quarantine_reason").isNotNull())

(
    clean.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.silver.airline")
)

(
    quarantined.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.silver.airline_quarantine")
)

In [0]:
booking_silver = (
    spark.read.table(f"{catalog}.bronze.booking")
    .select(
        F.col("bk_bookingid").alias("bk_bookingid"),
        F.col("bk_passengerid").cast("int").alias("bk_passengerid"),
        F.upper(F.trim(F.col("bk_status"))).alias("bk_status"),
        F.col("bk_totalprice").cast("double").alias("bk_totalprice"),
        F.try_to_date(F.col("bk_bookingdate")).alias("bk_bookingdate"),
        F.try_to_date(F.col("bk_flightdate")).alias("bk_flightdate"),
        F.upper(F.trim(F.col("bk_bookingpriority"))).alias("bk_bookingpriority"),
        F.initcap(F.trim(F.col("bk_agent"))).alias("bk_agent"),
        F.col("bk_comment").alias("bk_comment"),
        F.col("_source_filename"),
        F.col("_ingested_at"),
        F.col("_job_run_id")
    ).dropDuplicates()
)

w = Window.partitionBy("bk_bookingid").orderBy(F.col("_ingested_at").desc())
booking_ranked = booking_silver.withColumn("_pk_rank", F.row_number().over(w))

valid_passenger_ids = spark.table(f"{catalog}.silver.passenger").select(F.col("pa_passengerid").alias("_valid_passengerid"))

booking_flagged = (
    booking_ranked
    .join(valid_passenger_ids, F.col("bk_passengerid") == F.col("_valid_passengerid"), "left")
    .withColumn(
        "quarantine_reasons",
        F.array_compact(
            F.array(
                F.when(F.col("bk_bookingid").isNull(), "NULL primary key"),
                F.when(F.col("_pk_rank") > 1, "Duplicate primary key - older record"),
                F.when(F.col("bk_passengerid").isNull(), "NULL FK (bk_passengerid)"),
                F.when(F.col("_valid_passengerid").isNull(), "Foreign key not found in parent (silver.passenger)"),
                F.when(F.col("bk_totalprice") < 0, "Negative bk_total price"),
                F.when(
                    F.col("bk_bookingdate").isNull()
                    | (F.col("bk_bookingdate") < F.lit("1994-01-01").cast("date"))
                    | (F.col("bk_bookingdate") > F.lit("1998-12-31").cast("date")),
                    "Booking date outside operational window [1994-01-01, 1998-12-31]"
                ),
                F.when(
                    F.col("bk_flightdate").isNull()
                    | (F.col("bk_flightdate") < F.lit("1994-01-01").cast("date"))
                    | (F.col("bk_flightdate") > F.lit("1998-12-31").cast("date")),
                    "Flight date outside operational window [1994-01-01, 1998-12-31]"
                ),
                F.when(F.col("bk_flightdate") < F.col("bk_bookingdate"), "booking Date is after the flight date")
            )
        )
    )
    .drop("_pk_rank", "_valid_passengerid")
)

clean = (
    booking_flagged.filter(F.size(F.col("quarantine_reasons")) == 0)
    .drop("quarantine_reasons", "_ingested_at", "_source_filename", "_job_run_id")
)

quarantined = booking_flagged.filter(F.size(F.col("quarantine_reasons")) > 0)

(
    clean.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.silver.booking")
)

(
    quarantined.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.silver.booking_quarantine")
)

In [0]:
route_silver = (
    spark.read.table(f"{catalog}.bronze.route")
    .select(
        F.col("ro_routeid"),
        F.upper(F.trim(F.col("ro_name"))).alias("ro_name"),
        F.initcap(F.trim(F.col("ro_aircraft"))).alias("ro_aircraft"),
        F.initcap(F.trim(F.col("ro_airline_brand"))).alias("ro_airline_brand"),
        F.upper(F.trim(F.col("ro_routetype"))).alias("ro_routetype"),
        F.col("ro_capacity"),
        F.upper(F.trim(F.col("ro_cabinconfig"))).alias("ro_cabinconfig"),
        F.col("ro_baseprice"),
        F.trim(F.col("ro_comment")).alias("ro_comment"),
        F.col("_source_file"),
        F.col("_ingested_at"),
        F.col("_job_run_id"),
    ).dropDuplicates()
)


w = Window.partitionBy("ro_routeid").orderBy(F.col("_ingested_at").desc())
route_ranked = route_silver.withColumn("_pk_rank", F.row_number().over(w))

route_flagged = (
    route_ranked
    .withColumn(
        "quarantine_reasons",
        F.array_compact(
            F.array(
                F.when(F.col("ro_routeid").isNull(), "NULL primary key"),
                F.when(F.col("_pk_rank") > 1, "Duplicate primary key - older record"),
                F.when(F.col("ro_baseprice") <= 0, "Negative baseprice")
            )
        )
    )
    .drop("_pk_rank")
)

clean = (
    route_flagged.filter(F.size(F.col("quarantine_reasons")) == 0)
    .drop("quarantine_reasons", "_ingested_at", "_source_file", "_job_run_id")
)

quarantined = route_flagged.filter(F.size(F.col("quarantine_reasons")) > 0)

(
    clean.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.silver.route")
)

(
    quarantined.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.silver.route_quarantine")
)

# --- Report ---
total = route_silver.count()
clean_count = clean.count()
quarantine_count = quarantined.count()

print(f"Total records:      {total}")
print(f"✅ Clean → silver.route:               {clean_count}")
print(f"❌ Quarantined → silver.route_quarantine: {quarantine_count}")

if quarantine_count > 0:
    print("\nQuarantined records:")
    quarantined.display()
else:
    print("No violations detected — all records passed validation.")

# Show clean result
spark.table(f"{catalog}.silver.route").display()

In [0]:
airline_route_silver = (
    spark.read.table(f"{catalog}.bronze.airline_route")
    .select(
        F.col("ar_airlineid"),
        F.col("ar_routeid"),
        F.col("ar_available_seats"),
        F.col("ar_operatingcost"),
        F.trim(F.col("ar_comment")).alias("ar_comment"),
        F.col("_source_file"),
        F.col("_ingested_at"),
        F.col("_job_run_id"),
    ).dropDuplicates()
)

# PK validation: composite (ar_airlineid, ar_routeid) must be NOT NULL and UNIQUE
w = Window.partitionBy("ar_airlineid", "ar_routeid").orderBy(F.col("_ingested_at").desc())
ar_ranked = airline_route_silver.withColumn("_pk_rank", F.row_number().over(w))

# FK: ar_airlineid → silver.airline.al_airlineid
valid_airline_ids = spark.table(f"{catalog}.silver.airline").select(
    F.col("al_airlineid").alias("_valid_airlineid")
)

# FK: ar_routeid → silver.route.ro_routeid
valid_route_ids = spark.table(f"{catalog}.silver.route").select(
    F.col("ro_routeid").alias("_valid_routeid")
)

ar_flagged = (
    ar_ranked
    .join(valid_airline_ids, F.col("ar_airlineid") == F.col("_valid_airlineid"), "left")
    .join(valid_route_ids, F.col("ar_routeid") == F.col("_valid_routeid"), "left")
    .withColumn(
        "quarantine_reasons",
        F.array_compact(
            F.array(
                # --- PK checks ---
                F.when(
                    F.col("ar_airlineid").isNull() | F.col("ar_routeid").isNull(),
                    "NULL primary key (ar_airlineid, ar_routeid)"
                ),
                F.when(F.col("_pk_rank") > 1, "Duplicate primary key - older record"),
                # --- FK checks ---
                F.when(
                    F.col("ar_airlineid").isNull() | F.col("_valid_airlineid").isNull(),
                    "FK ar_airlineid not found in silver.airline"
                ),
                F.when(
                    F.col("ar_routeid").isNull() | F.col("_valid_routeid").isNull(),
                    "FK ar_routeid not found in silver.route"
                ),
                # --- Numeric measure validation (SkyLedger business rules) ---
                F.when(
                    F.col("ar_available_seats").isNull() | (F.col("ar_available_seats") < 0),
                    "Available seats must be non-negative"
                ),
                F.when(
                    F.col("ar_operatingcost").isNull() | (F.col("ar_operatingcost") < 0),
                    "Operating cost must be non-negative"
                ),
            )
        )
    )
    .drop("_pk_rank", "_valid_airlineid", "_valid_routeid")
)

clean = (
    ar_flagged.filter(F.size(F.col("quarantine_reasons")) == 0)
    .drop("quarantine_reasons", "_ingested_at", "_source_file", "_job_run_id")
)

quarantined = ar_flagged.filter(F.size(F.col("quarantine_reasons")) > 0)

(
    clean.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.silver.airline_route")
)

(
    quarantined.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.silver.airline_route_quarantine")
)

# --- Report ---
total = airline_route_silver.count()
clean_count = clean.count()
quarantine_count = quarantined.count()

print(f"Total records:          {total}")
print(f"✅ Clean → silver.airline_route:               {clean_count}")
print(f"❌ Quarantined → silver.airline_route_quarantine: {quarantine_count}")

if quarantine_count > 0:
    print("\nQuarantine reason breakdown:")
    quarantined.select(
        F.explode(F.col("quarantine_reasons")).alias("reason")
    ).groupBy("reason").count().orderBy(F.col("count").desc()).display()
    print("\nSample quarantined records:")
    quarantined.limit(10).display()
else:
    print("No violations detected — all records passed validation.")

# Show clean result
spark.table(f"{catalog}.silver.airline_route").limit(10).display()

In [0]:
flight_booking_silver = (
    spark.read.table(f"{catalog}.bronze.flight_booking")
    .select(
        F.col("fb_bookingid"),
        F.col("fb_routeid"),
        F.col("fb_airlineid"),
        F.col("fb_linenumber"),
        F.col("fb_seats"),
        F.col("fb_ticketprice"),
        F.col("fb_discount"),
        F.col("fb_tax"),
        F.upper(F.trim(F.col("fb_refundflag"))).alias("fb_refundflag"),
        F.upper(F.trim(F.col("fb_segmentstatus"))).alias("fb_segmentstatus"),
        F.col("fb_departuredate"),
        F.col("fb_commitdate"),
        F.col("fb_arrivaldate"),
        F.upper(F.trim(F.col("fb_cabinclass"))).alias("fb_cabinclass"),
        F.trim(F.col("fb_comment")).alias("fb_comment"),
        F.col("_rescued_data"),
        F.col("_source_filename"),
        F.col("_ingested_at"),
        F.col("_job_run_id"),
    ).dropDuplicates()
)

# PK validation: composite (fb_bookingid, fb_linenumber) must be NOT NULL and UNIQUE
w = Window.partitionBy("fb_bookingid", "fb_linenumber").orderBy(F.col("_ingested_at").desc())
fb_ranked = flight_booking_silver.withColumn("_pk_rank", F.row_number().over(w))

# FK: fb_bookingid → silver.booking.bk_bookingid (cast to string to match parent type)
# Also pull bk_bookingdate for cross-column check
valid_booking = spark.table(f"{catalog}.silver.booking").select(
    F.col("bk_bookingid").cast("string").alias("_valid_bookingid"),
    F.col("bk_bookingdate").alias("_parent_bookingdate"),
)

# FK: fb_routeid → silver.route.ro_routeid
valid_route = spark.table(f"{catalog}.silver.route").select(
    F.col("ro_routeid").alias("_valid_routeid")
)

# FK: fb_airlineid → silver.airline.al_airlineid
valid_airline = spark.table(f"{catalog}.silver.airline").select(
    F.col("al_airlineid").alias("_valid_airlineid")
)

fb_flagged = (
    fb_ranked
    .join(valid_booking, F.col("fb_bookingid").cast("string") == F.col("_valid_bookingid"), "left")
    .join(valid_route, F.col("fb_routeid") == F.col("_valid_routeid"), "left")
    .join(valid_airline, F.col("fb_airlineid") == F.col("_valid_airlineid"), "left")
    .withColumn(
        "quarantine_reasons",
        F.array_compact(
            F.array(
                # --- PK checks ---
                F.when(
                    F.col("fb_bookingid").isNull() | F.col("fb_linenumber").isNull(),
                    "NULL primary key (fb_bookingid, fb_linenumber)"
                ),
                F.when(F.col("_pk_rank") > 1, "Duplicate primary key - older record"),
                # --- FK checks ---
                F.when(
                    F.col("fb_bookingid").isNull() | F.col("_valid_bookingid").isNull(),
                    "FK fb_bookingid not found in silver.booking"
                ),
                F.when(
                    F.col("fb_routeid").isNull() | F.col("_valid_routeid").isNull(),
                    "FK fb_routeid not found in silver.route"
                ),
                F.when(
                    F.col("fb_airlineid").isNull() | F.col("_valid_airlineid").isNull(),
                    "FK fb_airlineid not found in silver.airline"
                ),
                # --- Date window: all dates must be within [1994-01-01, 1998-12-31] ---
                F.when(
                    F.col("fb_departuredate").isNull()
                    | (F.col("fb_departuredate") < F.lit("1994-01-01").cast("date"))
                    | (F.col("fb_departuredate") > F.lit("1998-12-31").cast("date")),
                    "Departure date outside operational window [1994-01-01, 1998-12-31]"
                ),
                F.when(
                    F.col("fb_commitdate").isNull()
                    | (F.col("fb_commitdate") < F.lit("1994-01-01").cast("date"))
                    | (F.col("fb_commitdate") > F.lit("1998-12-31").cast("date")),
                    "Commit date outside operational window [1994-01-01, 1998-12-31]"
                ),
                F.when(
                    F.col("fb_arrivaldate").isNull()
                    | (F.col("fb_arrivaldate") < F.lit("1994-01-01").cast("date"))
                    | (F.col("fb_arrivaldate") > F.lit("1998-12-31").cast("date")),
                    "Arrival date outside operational window [1994-01-01, 1998-12-31]"
                ),
                # --- Cross-column consistency ---
                F.when(
                    F.col("_parent_bookingdate").isNotNull()
                    & F.col("fb_departuredate").isNotNull()
                    & (F.col("fb_departuredate") < F.col("_parent_bookingdate")),
                    "Departure date earlier than parent booking date"
                ),
                F.when(
                    F.col("fb_departuredate").isNotNull()
                    & F.col("fb_arrivaldate").isNotNull()
                    & (F.col("fb_arrivaldate") < F.col("fb_departuredate")),
                    "Arrival date earlier than departure date"
                ),
                # --- Numeric measure validation (SkyLedger business rules) ---
                F.when(
                    F.col("fb_seats").isNull() | (F.col("fb_seats") <= 0),
                    "Seats must be a positive integer"
                ),
                F.when(
                    F.col("fb_ticketprice").isNull() | (F.col("fb_ticketprice") <= 0),
                    "Ticket price must be positive"
                ),
                F.when(
                    F.col("fb_discount").isNull() | (F.col("fb_discount") < 0) | (F.col("fb_discount") > 1),
                    "Discount must be between 0 and 1"
                ),
                F.when(
                    F.col("fb_tax").isNull() | (F.col("fb_tax") < 0) | (F.col("fb_tax") > 1),
                    "Tax must be between 0 and 1"
                ),
            )
        )
    )
    .drop("_pk_rank", "_valid_bookingid", "_parent_bookingdate", "_valid_routeid", "_valid_airlineid")
)

clean = (
    fb_flagged.filter(F.size(F.col("quarantine_reasons")) == 0)
    .drop("quarantine_reasons", "_rescued_data", "_ingested_at", "_source_filename", "_job_run_id")
)

quarantined = fb_flagged.filter(F.size(F.col("quarantine_reasons")) > 0)

(
    clean.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.silver.flight_booking")
)

(
    quarantined.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog}.silver.flight_booking_quarantine")
)


In [0]:
%sql
select count(*) from tpc_a_dev.silver.flight_booking_quarantine